# 06 — General concordance between RNA and ADT

In nb02, we showed that perturbation (CRISPR-KO) leads to downreg of RNA for that target.  For the 12 CRISPR targets that have ADT signal, we showed that KO also leads to downreg of ADT, but less than RNA level.  Some targets, like HLA's seem more or less unchanged at protein level, while they undergo -1.5 L2FC at RNA level. 

Here, looking more broadly at concordance between RNA and ADT signal (all 20 targets)

In [ ]:
# =============================================================================
# nb06 — RNA / ADT concordance and signature-level context dependence
#
# Two questions:
#   1. Does surface protein track transcript at the population level?
#      Baseline concordance in control-guide cells, then all cells.
#   2. Which perturbations have context-dependent effects — signatures that
#      diverge across conditions — and do those effects hold at the protein
#      level?
#
# The signature matrix from nb05 is the input for question 2.
# Question 1 is computed directly from the normalised expression objects.
# =============================================================================

%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
import muon as mu

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig

cfg    = load_config()
panels = load_panels()
P      = paths(cfg)
SEED   = set_seed(cfg)
apply_style(cfg)
sc.settings.verbosity = 1

s          = cfg["schema"]["obs"]
PERT       = s["perturbation"]
COND       = s["condition"]
CTRL       = cfg["schema"]["control_label"]
cond_order = ["Control", "IFNγ", "Co-culture"]
pal        = condition_palette(cfg)

# ---- data -----------------------------------------------------------------
import mudata as md
mdata = md.read(P.data_interim / "frangieh_qc.h5mu")
rna, adt = mdata["rna"], mdata["adt"]

# remove T-cell contamination identified in nb03
emb = pd.read_parquet(P.data_interim / "03_embedding.parquet")
tcell = emb.index[emb["cluster"].astype(str) == "12"]
keep  = ~rna.obs_names.isin(tcell)
rna, adt = rna[keep].copy(), adt[keep].copy()

# ---- signature matrix from nb05 ------------------------------------------
sig  = pd.read_parquet(P.data_processed / "05_signatures_lfc.parquet")
padj = pd.read_parquet(P.data_processed / "05_signatures_padj.parquet")

# ---- normalise both modalities -------------------------------------------
# RNA: log-normalised counts
rna_n = rna.copy()
rna_n.X = rna_n.layers["counts"].copy()
sc.pp.normalize_total(rna_n, target_sum=1e4)
sc.pp.log1p(rna_n)

# ADT: CLR per cell
adt_n = adt.copy()
adt_n.layers["counts"] = adt_n.X.copy()
mu.prot.pp.clr(adt_n, axis=cfg["protein"]["clr_margin"])

# panel metadata
adt_to_rna = panels["adt"]["adt_to_rna"]
isotypes   = panels["adt"]["isotype_controls"]
targets    = [f for f in adt_n.var_names if f not in isotypes]
adt_annot  = panels["adt"]["annotations"]

print(f"RNA:     {rna_n.n_obs:,} cells x {rna_n.n_vars:,} genes")
print(f"ADT:     {adt_n.n_obs:,} cells x {len(targets)} targets + {len(isotypes)} isotypes")
print(f"sig:     {sig.shape[0]} contrasts x {sig.shape[1]} genes")
assert (rna_n.obs_names == adt_n.obs_names).all(), "modalities out of order"